# Can we predict 30-day readmission?
Data: `data.csv` (one row per hospital stay), dictionary in `columns.csv`. Target: `readmit_30d`.

Key design point: the same person appears in many rows (stays). Random row-level splits would put the same person in train and test and inflate scores, so **all evaluation is grouped by `person_key`**.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.model_selection import StratifiedGroupKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, roc_curve
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance
pd.set_option('display.width',150)
df=pd.read_csv('data.csv'); print(pd.read_csv('columns.csv').to_string(index=False))
df.head()

## 1. Data checks

In [ ]:
print(df.shape, '| people:',df.person_key.nunique(), '| missing:',int(df.isna().sum().sum()), '| duplicate rows:',df.duplicated().sum())
print('stays per person:', df.groupby('person_key').size().describe().round(1).to_dict())
print('readmission rate:', df.readmit_30d.mean().round(3))
for col in ['birth_year','sex','postcode_area']:
    print(col,'varies within person for', (df.groupby('person_key')[col].nunique()>1).sum(),'people')
df.describe().T

## 2. Univariate relationships with readmission

In [ ]:
num=['visit_number','birth_year','postcode_area','bmi','length_of_stay_days','n_medications','lab_a','lab_b']
fig,ax=plt.subplots(2,4,figsize=(16,7))
for a,col in zip(ax.ravel(),num):
    q=pd.qcut(df[col],10,duplicates='drop')
    g=df.groupby(q,observed=True).readmit_30d.mean()
    a.plot(range(len(g)),g.values,'o-'); a.axhline(df.readmit_30d.mean(),ls='--',c='grey')
    a.set_title(col); a.set_xlabel('decile'); a.set_ylabel('readmit rate')
plt.tight_layout(); plt.show()
print(df.groupby('sex').readmit_30d.mean().round(3))

Person-level clustering: do some people get readmitted more often than chance? That would mean person history is informative, but it is only usable if available *before* the stay in question.

In [ ]:
g=df.groupby('person_key').readmit_30d.agg(['sum','size'])
p=df.readmit_30d.mean()
sim=np.random.default_rng(0).binomial(g['size'],p)
print('variance of per-person readmit count: observed %.2f vs. expected if independent %.2f'%(g['sum'].var(), pd.Series(sim).var()))

## 3. Modelling
Features are only those known by discharge of the stay; `person_key` is excluded as an identifier. Models: prior-rate baseline, logistic regression, gradient boosting. 5-fold cross-validation, stratified and grouped by person, so no person is in both train and test.

In [ ]:
feats=['visit_number','birth_year','sex','postcode_area','bmi','length_of_stay_days','n_medications','lab_a','lab_b']
X=df[feats].copy(); X['sex']=(X.sex=='M').astype(int); y=df.readmit_30d.values; groups=df.person_key.values
cv=StratifiedGroupKFold(5,shuffle=True,random_state=0)
folds=list(cv.split(X,y,groups))
models={'baseline (prior)':DummyClassifier(strategy='prior'),
 'logistic regression':make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000)),
 'gradient boosting':HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0)}
oof={}; rows=[]
for n,m in models.items():
    pr=cross_val_predict(m,X,y,groups=groups,cv=folds,method='predict_proba')[:,1]; oof[n]=pr
    aucs=[roc_auc_score(y[t],pr[t]) for _,t in folds]
    rows.append(dict(model=n,AUC=roc_auc_score(y,pr),AUC_fold_sd=np.std(aucs),PR_AUC=average_precision_score(y,pr),Brier=brier_score_loss(y,pr)))
res=pd.DataFrame(rows).set_index('model').round(3); print('prevalence (PR-AUC baseline): %.3f'%y.mean()); res

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4.5))
for n in list(models)[1:]:
    f,t,_=roc_curve(y,oof[n]); ax[0].plot(f,t,label=n+' (AUC %.3f)'%res.loc[n,'AUC'])
    fp,mp=calibration_curve(y,oof[n],n_bins=10,strategy='quantile'); ax[1].plot(mp,fp,'o-',label=n)
ax[0].plot([0,1],[0,1],'k--'); ax[0].set_xlabel('FPR'); ax[0].set_ylabel('TPR'); ax[0].legend(); ax[0].set_title('ROC (out-of-fold, grouped)')
ax[1].plot([0,.5],[0,.5],'k--'); ax[1].set_xlabel('mean predicted'); ax[1].set_ylabel('observed rate'); ax[1].legend(); ax[1].set_title('Calibration')
plt.tight_layout(); plt.show()

## 4. What drives the prediction?

In [ ]:
lr=models['logistic regression'].fit(X,y)
coef=pd.Series(lr[-1].coef_[0],index=feats).sort_values(); print('Standardised logistic coefficients (log-odds per 1 SD):'); print(coef.round(3))
imp=np.zeros(len(feats))
for tr,te in folds:
    m=HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0).fit(X.iloc[tr],y[tr])
    imp+=permutation_importance(m,X.iloc[te],y[te],scoring='roc_auc',n_repeats=5,random_state=0).importances_mean/5
pd.Series(imp,index=feats).sort_values().plot.barh(title='Permutation importance (AUC drop, held-out)'); plt.show()

## 5. Robustness: person-level bootstrap CI and shuffled-label check

In [ ]:
rng=np.random.default_rng(1); pr=oof['logistic regression']
people=np.unique(groups); idx=pd.Series(np.arange(len(y))).groupby(groups).apply(np.array)
bs=[]
for _ in range(500):
    s=np.concatenate(idx[rng.choice(people,len(people))].values); bs.append(roc_auc_score(y[s],pr[s]))
print('Logistic AUC 95%% CI (person bootstrap): %.3f - %.3f'%tuple(np.percentile(bs,[2.5,97.5])))
ys=pd.Series(y).groupby(groups).transform(lambda s: rng.permutation(s.values)).values
ps=cross_val_predict(models['logistic regression'],X,ys,groups=groups,cv=folds,method='predict_proba')[:,1]
print('AUC with shuffled labels: %.3f (should be ~0.5)'%roc_auc_score(ys,ps))

## 6. Practical utility: flagging the highest-risk stays

In [ ]:
d=pd.DataFrame({'p':oof['logistic regression'],'y':y})
rows=[]
for k in [0.05,0.1,0.2,0.3]:
    top=d.nlargest(int(k*len(d)),'p'); rows.append(dict(flag_top='%d%%'%(k*100),precision=top.y.mean(),lift=top.y.mean()/y.mean(),recall=top.y.sum()/y.sum()))
pd.DataFrame(rows).round(3)

## 7. Adding the person's own readmission history
Section 2 showed readmissions cluster within people far beyond chance, so a person's *earlier* outcomes may be predictive. We build `prior_readmits` / `prior_rate` from stays with a lower `visit_number` only (never the current stay's label), and keep grouped CV.

**Caveat:** this assumes `visit_number` is chronological and that earlier stays' outcomes are known when the prediction is made. A stay's 30-day label is only final 30 days after discharge, so at the time of a new admission the immediately preceding label may not be settled in practice. Confirm this with whoever owns the data before relying on this model.

In [ ]:
d2=df.sort_values(['person_key','visit_number']).copy()
d2['prior_readmits']=d2.groupby('person_key').readmit_30d.cumsum()-d2.readmit_30d
d2['prior_rate']=d2.prior_readmits/(d2.visit_number-1).clip(lower=1)
print('gaps in visit sequence:', int(d2.groupby('person_key').visit_number.apply(lambda s:(s.diff().dropna()!=1).sum()).sum()))
X2=d2[feats].copy(); X2['sex']=(X2.sex=='M').astype(int); X2['prior_readmits']=d2.prior_readmits; X2['prior_rate']=d2.prior_rate
y2=d2.readmit_30d.values; g2=d2.person_key.values
folds2=list(StratifiedGroupKFold(5,shuffle=True,random_state=0).split(X2,y2,g2))
rows=[]
for n in ['logistic regression','gradient boosting']:
    pr=cross_val_predict(models[n],X2,y2,groups=g2,cv=folds2,method='predict_proba')[:,1]
    rows.append(dict(model=n+' + history',AUC=roc_auc_score(y2,pr),PR_AUC=average_precision_score(y2,pr),Brier=brier_score_loss(y2,pr)))
pd.DataFrame(rows).set_index('model').round(3)